In [ ]:
%pip install pandas numpy matplotlib scikit-learn

#Alzheimer Detection using Machine Learning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

ModuleNotFoundError: No module named 'pandas'

In [ ]:
%pip install openpyxl

#Load dataset
diagnosis_df = pd.read_excel("CSI_7_MAL_2324_CW_resit_data.xlsx", sheet_name="Diagnosis target")    
diagnosis_df.head()

In [ ]:
cognitive_scores_df = pd.read_excel("CSI_7_MAL_2324_CW_resit_data.xlsx", sheet_name="Cognitive score")
cognitive_scores_df.head()

In [ ]:
#load the data sheet
data_df = pd.read_excel('CSI_7_MAL_2324_CW_resit_data.xlsx', sheet_name='Data')
data_df.head()

In [ ]:
# fix missing values in each DataFrame
diagnosis_df.dropna(inplace=True)
cognitive_scores_df.dropna(inplace=True)
data_df.dropna(inplace=True)

In [ ]:
# merge DataFrames on 'RID' column
merged_df = pd.merge(diagnosis_df, cognitive_scores_df, on='RID', how='inner')
merged_df = pd.merge(merged_df, data_df, on='RID', how ='inner')
merged_df.head()

In [ ]:
# check the shape of the merged DataFrame
merged_df.shape

In [ ]:
# standardize numerical columns
numerical_cols = merged_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
scaler = StandardScaler()
merged_df[numerical_cols] = scaler.fit_transform(merged_df[numerical_cols])

In [ ]:
#feature selection time
selected_features = merged_df.drop(columns=['RID', 'Test_data'])
#print the selected features
print("Selected features:", selected_features.columns)

In [ ]:
#encode categorical variables
for cols in selected_features.columns:
    if selected_features[cols].dtype == 'object':
        selected_features[cols] = pd.Categorical(selected_features[cols]).codes
        
        selected_features.head()

In [ ]:
#nan values with mean will be pkaced in FDG column
selected_features["FDG_PET"] = selected_features["FDG_PET"].fillna(selected_features["FDG_PET"].mean())

In [ ]:
selected_features.isnull().sum()

In [ ]:
#model training
#define mapping of categories to desired order
category_mapping = { 
    "CN": "Cognitively Normal",
    "AD" : "Alzheimer's Disease",
    "LMCI" : "Mild Cognitive Impairment"}
#map the categories to the desired order
selected_features['Diagnosis'] = selected_features['Diagnosis'].map(category_mapping)

In [ ]:
#split data into features (x) and target (y)
x = selected_features.drop(columns=['Diagnosis'])
y = selected_features['Diagnosis']

In [ ]:
#encoding the target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
#split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(
    x, y_encoded, test_size=0.2, random_state=42)

In [ ]:
#Initialize Random Forest Classifier
rf_classifier = RandomForestClassifier()

#Train the model
rf_classifier.fit(x_train, y_train)

In [ ]:
#evaluate the model
training_accuracy = rf_classifier.score(x_train, y_train)
testing_accuracy = rf_classifier.score(x_test, y_test)
print(f"Training Accuracy: {training_accuracy*100:.2f}%")
print(f"Testing Accuracy: {testing_accuracy*100:.2f}%")